In [6]:
puts `ls -lt ./maps/`

total 868
-rw-rw-r-- 1 osboxes osboxes  84435 Apr 18 09:53 2026-drug-mapping-errors.txt
-rw-rw-r-- 1 osboxes osboxes 103903 Apr 18 09:53 2026-drug-mappings.map
drwxrwxr-x 2 osboxes osboxes   4096 Apr 18 09:03 deprecated
-rw-rw-r-- 1 osboxes osboxes   4830 Apr 17 13:36 2026-gene-errors.txt
-rw-rw-r-- 1 osboxes osboxes 380756 Apr 17 13:36 2026-gene-mappings.map
-rw-rw-r-- 1 osboxes osboxes   8814 Apr 17 10:46 2026-unmapped-cuis.csv
-rw-rw-r-- 1 osboxes osboxes 275711 Apr 17 10:46 2026-demokritos-disease-mondo.map


In [7]:
puts `head -2 ./maps/2026-drug-mappings.map`

demokritosid,xref,pubchem_cid,IUPACname
C0000376,http://purl.bioontology.org/ontology/MESH/D015102,https://pubchem.ncbi.nlm.nih.gov/compound/547,"3,4-Dihydroxyphenylacetic Acid"


In [8]:
puts `head -2 ./maps/2026-demokritos-disease-mondo.map`

demokritos_umls,prefname,mondo
C0000744,abetalipoproteinemia,http://purl.obolibrary.org/obo/MONDO_0008692


In [9]:
puts `head -2 "./raw-data/Drug-Disease triples.tsv"`
puts `cat "./raw-data/Drug-Disease triples.tsv" | wc -l`
puts
puts `head -2 "./raw-data/Disease-Drug triples.tsv"`
puts `cat "./raw-data/Disease-Drug triples.tsv" | wc -l`

Drug	Drug_id	RELATION	PROVENANCE	Disease	Disease_id
Lysine	C0024337	CAUSES	["23658800_fullText_123"]	Hypoglycemia	C0020615
857

Disease	Disease_id	RELATION	PROVENANCE	Drug	Drug_id
Glutaric aciduria, type 1	C0268595	PRODUCES	["20032085_fullText_1"]	Lysine	C0024337
1286


In [ ]:
require 'linkeddata'
require 'csv'

graphing_errors = File.open('./graph/2026-drug-disease-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
drug_mappings = CSV.read('./maps/2026-drug-mappings.map', headers: true)
disease_mappings = CSV.read('./maps/2026-demokritos-disease-mondo.map', headers: true)

# Create RDF graph
graph = RDF::Repository.new  # small enough to be held in memory

failures = {}
# Process each entity relation
recordcount = 0
['./raw-data/Drug-Disease triples.tsv', './raw-data/Disease-Drug triples.tsv'].each do |sourcefile|
  CSV.foreach(sourcefile, col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|  # warn row.inspect
      print "#{recordcount}, " 
      recordcount=recordcount + 1
      
      drug_id = row['Drug_id']
      disease_id = row['Disease_id']
      evidence = row['PROVENANCE']
      evidence_url = nil
      if evidence =~ /(\d+)_\w+_/
          evidence_url = "https://pubmed.ncbi.nlm.nih.gov/#{$1}"  # use short form for matches
      end

      source_relation = row['RELATION']

      # Find corresponding mappings
      drug = drug_mappings.find { |d| d['demokritosid'] == drug_id }
      disease = disease_mappings.find { |d| d['demokritos_umls'] == disease_id }
      
      unless drug
        next if failures[drug_id]
        failures[drug_id] = 1
        # warn "drug lookup failed #{drug_id}"
        graphing_errors.write "drug lookup failed #{drug_id}\n"
        next
      end
      unless disease
        next if failures[disease_id]
        failures[disease_id] = 1
        # warn "disease lookup failed #{disease_id}"
        graphing_errors.write "disease lookup failed #{disease_id}\n"
        next
      end
      
      # Extract relevant IDs and labels
      # demokratisid,xref,demokratis_label,pubchem_cid,IUPACname
      # C0613621,http://purl.bioontology.org/ontology/MESH/C030536,"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether",https://pubchem.ncbi.nlm.nih.gov/compound/152803,"2,2-dichloro-1,1-difluoroethyl%20difluoromethyl%20ether"
      pubchem_uri = RDF::URI.new(drug['pubchem_cid'])
      pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
      pubchem_label =  RDF::Literal.new(drug['xref'])
      pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
      # human_drug_label = RDF::Literal.new(drug['demokritos_label'])
      iupac_drug_label = RDF::Literal.new(drug['IUPACname'])
        
    # demokritos_umls,prefname,mondo
    # C0000774,gastrin secretion abnormality,http://purl.obolibrary.org/obo/MONDO_0001770
      mondo_uri = RDF::URI.new(disease['mondo'])
      mondo_type = RDF::URI.new("https://bioportal.bioontology.org/ontologies/MONDO")
      mondo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Disease")
      mondo_label =  RDF::Literal.new("MONDO Term")
    #   orphanet = RDF::URI.new(disease['orpha'])
      disease_label = RDF::Literal.new(disease['prefname'])
      original_disease = RDF::Literal.new(disease_id)
      
      # Create context URI
      #context_uri = RDF::URI.new("urn:simpathic:context:dem_#{drug_id}_#{disease_id}")
      general_context = RDF::URI.new("urn:simpathic:context:all_metadata")


      if sourcefile =~ /Drug\-Disease/
        context_uri = RDF::URI.new("urn:simpathic:context:dem_#{drug_id}_#{disease_id}")
          graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)
      elsif sourcefile =~ /Disease\-Drug/
        context_uri = RDF::URI.new("urn:simpathic:context:dem_#{disease_id}_#{drug_id}")
          graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
      else
          abort "filename matching for directionality failed"
      end      
      # graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     human_drug_label, graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     iupac_drug_label, graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_type,          graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_core_type,             graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_type, RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_core_type, RDFS.label,     RDF::Literal.new("Drug"), graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{drug_id}"), graph_name: context_uri)
      
        
      graph << RDF::Statement.new(mondo_uri, RDFS.label, disease_label, graph_name: context_uri)
      graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_type, graph_name: context_uri)
      graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_core_type, graph_name: context_uri)
      graph << RDF::Statement.new(mondo_type, RDFS.label, mondo_label, graph_name: context_uri)
    #   graph << RDF::Statement.new(mondo_uri, SIMPATHIC['orphanet'], orphanet, graph_name: context_uri)
      graph << RDF::Statement.new(mondo_uri, SIMPATHIC['original-id'], original_disease, graph_name: context_uri)
    
      graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Demokritos"), graph_name: general_context)
    # evidence = row['PROVENANCE']
    # source_relation = row['RELATION']
    graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'], 
                                RDF::URI.new(evidence_url)) if evidence_url
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],
                                RDF::Literal.new(source_relation))
    #   graph << RDF::Statement.new(context_uri, SIMPATHIC['score'], RDF::Literal.new(score))
    end
end
# Write RDF to file in N-Quads format
File.open('./graph/2026-demokritos_drug-disease.nq.large', 'w') do |f|
  RDF::Writer.for(:nquads).new(f) do |writer|
      warn "writing triples"
    writer << graph
  end
end
graphing_errors.close

puts "RDF quads written"

In [18]:
graphing_errors.close